# Boltz-2 screen for human GHS-R\n\nThis notebook runs the end-to-end GHS-R ligand screening workflow.

In [ ]:
!pip install -U boltz pandas matplotlib pyyaml py3Dmol

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nimport os\nos.environ['BOLTZ_CACHE'] = '/content/drive/MyDrive/boltz_cache'\nos.makedirs(os.environ['BOLTZ_CACHE'], exist_ok=True)\nprint('BOLTZ_CACHE =', os.environ['BOLTZ_CACHE'])

In [ ]:
# If this repo is on GitHub, clone it. Otherwise upload files manually to /content.\n# !git clone <your-repo-url> /content/ghsr_af3_molecule_screening\n%cd /content/ghsr_af3_molecule_screening

In [ ]:
!python scripts/build_inputs.py --library data/ligand_library.csv --target-fasta data/target_ghsr.fasta --out-dir inputs

In [ ]:
!python scripts/run_screen.py --inputs-dir inputs --outputs-dir outputs --recycling-steps 3 --diffusion-samples 1 --msa-mode server

In [ ]:
!python scripts/parse_results.py --library data/ligand_library.csv --outputs-dir outputs --results-dir results\n!python scripts/visualize.py --ranking-csv results/ranking.csv --outputs-dir outputs --plots-dir results/plots --top-poses-dir results/top_poses

In [ ]:
import pandas as pd\ndf = pd.read_csv('results/ranking.csv')\ndf.head(20)

In [ ]:
import py3Dmol\nfrom pathlib import Path\n\ntop_pose_dir = Path('results/top_poses')\nposes = sorted(top_pose_dir.glob('*.pdb'))\nprint('Found', len(poses), 'pose files')\nif poses:\n    model = poses[0].read_text()\n    view = py3Dmol.view(width=900, height=600)\n    view.addModel(model, 'pdb')\n    view.setStyle({'cartoon': {'color': 'spectrum'}})\n    view.setStyle({'resn': ['LIG']}, {'stick': {}})\n    view.zoomTo()\n    view.show()